# Email Support 7B Model — Fine-tuning v2 (Robust & Multilingual)

**Target:** Qwen 2.5 7B
**Data:** 40 augmented pairs (Finnish + English)
**Goal:** Force memorization of ArtCloud support manuals

### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Upload **`7b_kb_v2.jsonl`** (from `data/training/`)


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
print('Done.')

In [ ]:
from google.colab import files
uploaded = files.upload() # Upload 7b_kb_v2.jsonl

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

In [ ]:
from datasets import load_dataset
import json

raw = load_dataset('json', data_files='7b_kb_v2.jsonl', split='train')

def format_record(row):
    messages = [
        {"role": "system",    "content": row['instruction']},
        {"role": "user",      "content": row['input']},
        {"role": "assistant", "content": row['output']},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = raw.map(format_record, remove_columns=raw.column_names)
print('Dataset ready.')

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = 'text',
        max_seq_length              = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 40,   # Increased significantly for memorization
        learning_rate               = 2e-4, # Slightly higher to learn faster
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 1,
        optim                       = 'adamw_8bit',
        seed                        = 42,
        output_dir                  = '/content/checkpoints',
    ),
)

trainer.train()

## Final Step — Save to Google Drive
**Mount Drive using the sidebar first!**

In [ ]:
import os
local_output = "/content/temp_v2"
print("Saving GGUF locally...")

model.save_pretrained_gguf(
    local_output,
    tokenizer,
    quantization_method = "q4_k_m"
)

from google.colab import drive
if not os.path.exists("/content/drive"):
    drive.mount('/content/drive')

print("Moving to Drive...")
!mv /content/temp_v2_gguf/*.gguf /content/drive/MyDrive/
print("Done! Download the new GGUF and run load_7b_model.sh")